In [3]:
%%bash
cd homework-05

git clone --depth 1 https://github.com/DataTalksClub/machine-learning-zoomcamp.git /tmp/zoomcamp 2>/dev/null || true
cp -a /tmp/zoomcamp/cohorts/2026/homework/05-deployment/. .
rm -rf /tmp/zoomcamp

if ! command -v uv &> /dev/null; then
    curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"

echo "=== Checksum pipeline.bin ==="
sha256sum pipeline.bin

echo "=== Question 1: uv version ==="
uv --version

uv sync --locked
uv run python smoke_test.py

=== Checksum pipeline.bin ===
1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin
=== Question 1: uv version ===
uv 0.13.6 (x86_64-unknown-linux-gnu)


Resolved 19 packages in 1ms
Checked 18 packages in 0.23ms


{'conversion_probability': 0.769799, 'conversion': True}


In [4]:
%%bash
cd homework-05
echo "=== Question 2: pyproject.toml ==="
cat pyproject.toml
grep -A 3 'name = "scikit-learn"' uv.lock

=== Question 2: pyproject.toml ===
[project]
name = "mlzoomcamp-2026-deployment"
version = "0.1.0"
description = "Reference deployment project for ML Zoomcamp 2026"
requires-python = ">=3.11,<3.12"
dependencies = [
    "fastapi==0.119.1",
    "numpy==2.3.3",
    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",
]

[tool.uv]
exclude-newer = "2026-09-14T23:59:59Z"
    { name = "scikit-learn" },
    { name = "uvicorn" },
]

--
    { name = "scikit-learn", specifier = "==1.7.2" },
    { name = "uvicorn", specifier = "==0.38.0" },
]

--
name = "scikit-learn"
version = "1.7.2"
source = { registry = "https://pypi.org/simple" }
dependencies = [


In [5]:
import pickle
import sys
from pathlib import Path

sys.path.insert(0, str(Path("homework-05").resolve()))
from model import normalize_record

with open("homework-05/pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

record = normalize_record(lead)
prob = float(pipeline.predict_proba([record])[0, 1])
print(f"Question 3 conversion_probability: {prob:.3f}")

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9

Question 3 conversion_probability: 0.533


In [6]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
cd homework-05

uv run uvicorn predict:app --host 0.0.0.0 --port 9696 > /tmp/uvicorn.log 2>&1 &
SERVER_PID=$!
sleep 3

echo "=== Question 4 response ==="
curl -s http://localhost:9696/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
  }'
echo ""

kill $SERVER_PID
wait $SERVER_PID 2>/dev/null || true

=== Question 4 response ===
{"conversion_probability":0.769799,"conversion":true}


In [7]:
%%bash
cd homework-05
echo "=== Question 5: Dockerfile ==="
cat Dockerfile

=== Question 5: Dockerfile ===
FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]


In [9]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
cd homework-05

# 1. Usuń ewentualne stare śmieci po poprzednim uruchomieniu
docker rm -f hw5-container 2>/dev/null || true

# 2. Uruchom nowy kontener
docker run -d --rm --name hw5-container -p 9696:9696 zoomcamp-model:2026-hw5

# 3. Daj mu 5 sekund na pełny rozruch i uvcorn startup
echo "Czekam na uruchomienie serwera w kontenerze..."
sleep 5

# 4. Sprawdź czy kontener w ogóle żyje
if ! docker ps | grep -q hw5-container; then
    echo "BŁĄD: Kontener przedwcześnie się zatrzymał! Oto jego logi:"
    docker logs hw5-container
    exit 1
fi

echo "=== Health check z kontenera ==="
curl -s http://localhost:9696/health
echo ""

echo "=== Question 6 test ==="
uv run python q6_test.py

# 5. Posprzątaj po sobie
docker stop hw5-container

f59650e5bc7001b0f10a23760e0bc0f8cd34a762feedfe336a2f4961622daaf8
Czekam na uruchomienie serwera w kontenerze...
=== Health check z kontenera ===
{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}
=== Question 6 test ===
{'conversion_probability': 0.769799, 'conversion': True}
hw5-container
